In [1]:
!pip install qiskit qiskit-aer qiskit-nature qiskit-algorithms pyscf matplotlib pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 35.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 57.9 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 14.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.6 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from pyscf import gto, scf, mcscf, cc

HARTREE_TO_EV = 27.211386245988
EXPERIMENTAL_PI_PISTAR_EV = 7.6
np.random.seed(1)

In [3]:
def rotate_z(xyz, angle_deg):
    a = np.radians(angle_deg)
    R = np.array([[np.cos(a), -np.sin(a), 0],
                  [np.sin(a),  np.cos(a), 0],
                  [0, 0, 1]])
    return R @ np.array(xyz)

def build_geometry(twist_deg: float = 0.0) -> str:
    atoms = {k: v.copy() for k, v in equilibrium_atoms.items()}
    if twist_deg != 0.0:
        atoms["H3"] = rotate_z(atoms["H3"], twist_deg)
        atoms["H4"] = rotate_z(atoms["H4"], twist_deg)
    order = ["C1", "C2", "H1", "H2", "H3", "H4"]
    return "; ".join(f"{name[0]} {x:.6f} {y:.6f} {z:.6f}" for name in order
                      for x, y, z in [atoms[name]])

def print_geometry(label, geometry):
    print(f"{label}:")
    atoms = geometry.split("; ")
    for i, atom in enumerate(atoms):
        suffix = ";" if i < len(atoms) - 1 else ""
        print(f"  {atom}{suffix}")

# Standard planar equilibrium ethylene geometry (Angstrom)
equilibrium_atoms = {
    "C1": np.array([0.0000,  0.0000,  0.6695]),
    "C2": np.array([0.0000,  0.0000, -0.6695]),
    "H1": np.array([0.0000,  0.9289,  1.2321]),
    "H2": np.array([0.0000, -0.9289,  1.2321]),
    "H3": np.array([0.0000,  0.9289, -1.2321]),
    "H4": np.array([0.0000, -0.9289, -1.2321]),
}


geometry_equilibrium = build_geometry(0.0)
geometry_twisted = build_geometry(90.0)

# print(geometry_equilibrium)
print_geometry("Equilibrium", geometry_equilibrium)
print_geometry("\nTwisted 90", geometry_twisted)

Equilibrium:
  C 0.000000 0.000000 0.669500;
  C 0.000000 0.000000 -0.669500;
  H 0.000000 0.928900 1.232100;
  H 0.000000 -0.928900 1.232100;
  H 0.000000 0.928900 -1.232100;
  H 0.000000 -0.928900 -1.232100

Twisted 90:
  C 0.000000 0.000000 0.669500;
  C 0.000000 0.000000 -0.669500;
  H 0.000000 0.928900 1.232100;
  H 0.000000 -0.928900 1.232100;
  H -0.928900 0.000000 -1.232100;
  H 0.928900 -0.000000 -1.232100


In [4]:
mol = gto.M(atom=geometry_equilibrium.replace("; ", "\n"), basis="sto-3g",
            charge=0, spin=0, unit="Angstrom")
mf = scf.RHF(mol).run(verbose=0)

# CASCI(2,2): Increase to 3 states to capture Ground [0], Triplet [1], and Singlet [2]
mc_casci = mcscf.CASCI(mf, 2, 2)
mc_casci.verbose = 0
mc_casci.fcisolver.nstates = 3
e_casci_states = mc_casci.kernel()[0]
casci_ground = e_casci_states[0]

# Extract the singlet (index 2) instead of the triplet (index 1)
casci_excitation_eV = (e_casci_states[2] - e_casci_states[0]) * HARTREE_TO_EV

# CASSCF(2,2): State-average over 3 states to optimize orbitals for the singlet
mc_casscf = mcscf.CASSCF(mf, 2, 2)
mc_casscf = mc_casscf.state_average_([0.33, 0.33, 0.34])
mc_casscf.verbose = 0
mc_casscf.kernel()
casscf_ground = mc_casscf.e_states[0]
casscf_excitation_eV = (mc_casscf.e_states[2] - mc_casscf.e_states[0]) * HARTREE_TO_EV

# EOM-CCSD: Already correct because it explicitly targets singlets
mycc = cc.CCSD(mf).run(verbose=0)
eom_excitations_eV = np.array(mycc.eomee_ccsd_singlet(nroots=2)[0]) * HARTREE_TO_EV

print(f"RHF               ground energy: {mf.e_tot:.6f} Ha")
print(f"CASCI(2,2)        ground energy: {casci_ground:.6f} Ha   |  S0->S1 excitation: {casci_excitation_eV:.3f} eV")
print(f"CASSCF(2,2), SA   ground energy: {casscf_ground:.6f} Ha   |  S0->S1 excitation: {casscf_excitation_eV:.3f} eV")
print(f"EOM-CCSD (full)   ground energy: {mycc.e_tot:.6f} Ha   |  S0->S1 excitation: {eom_excitations_eV[0]:.3f} eV")
print(f"                                                          S0->S2 excitation: {eom_excitations_eV[1]:.3f} eV")

RHF               ground energy: -77.072088 Ha
CASCI(2,2)        ground energy: -77.116593 Ha   |  S0->S1 excitation: 14.155 eV
CASSCF(2,2), SA   ground energy: -77.116475 Ha   |  S0->S1 excitation: 14.142 eV
EOM-CCSD (full)   ground energy: -77.234131 Ha   |  S0->S1 excitation: 11.609 eV
                                                          S0->S2 excitation: 12.921 eV
